In [ ]:
import requests
import pandas as pd

from shapely.geometry import Point
from geopandas import GeoDataFrame

In [ ]:
server = "https://data.epa.ie/bw/api/v1/locations?per_page=500"

In [ ]:
response = requests.get(server)
data = response.json()
data

In [ ]:
rawData = pd.json_normalize(data['list'])
rawData = rawData[[
    "beach_id", "beach_name", "easting", "northing"
]]
rawData

In [ ]:
dataSet = []
for item in data['list']:
    if item['current_annual_water_quality_classification']:
        currentClass =  item['current_annual_water_quality_classification']
    else:
        currentClass = item['year1_annual_water_quality_classification']

    # Easterling, Northerling


    dataSet.append([
        item['beach_id'],
        item['beach_name'],
        item['easting'],
        item['northing']
    ])


In [ ]:
IrelandDF = pd.DataFrame(dataSet, columns=["id", "name", "northing", "easting"]).set_index('id')


In [ ]:
geometry = [Point(x, y) for x, y in zip(IrelandDF['easting'], IrelandDF['northing'])]
for item in geometry:
    print(item.x, item.y)

In [ ]:
gdf = GeoDataFrame(IrelandDF, geometry=geometry).set_crs(29903)   
gdf

In [ ]:
gdf.crs

In [ ]:
gdf = gdf.to_crs(4326)

In [ ]:
lat, lon = [item.y for item in gdf['geometry']], [item.x for item in gdf['geometry']]


In [ ]:
gdf['lat'] = lat
gdf['lon'] = lon

In [ ]:
gdf = gdf.drop(columns=['easting', 'northing', 'geometry'])
gdf['alternate_name'] = gdf['name']

In [ ]:
locations = gdf[[
    'name', 'alternate_name', 'lat', 'lon'
]]
locations